# adaptmem train+test on Colab — v2 (subprocess, no indent issues)

1. Sol panel → Files → upload `longmemeval_s_cleaned.json`
2. Runtime → Run all
3. Bittikten sonra `results_ft100_400.json` indirilir

In [ ]:
!pip install -q sentence-transformers numpy

In [ ]:
import os; os.makedirs('adaptmem', exist_ok=True); os.makedirs('benchmarks/data', exist_ok=True); print('ok')

In [ ]:
%%writefile adaptmem/__init__.py
"""adaptmem — domain adaptation for retrieval, in five lines."""
from adaptmem.core import AdaptMem
from adaptmem.miner import HardNegativeMiner
from adaptmem.types import LabelledQuery, RetrievalHit

__all__ = ["AdaptMem", "HardNegativeMiner", "LabelledQuery", "RetrievalHit"]
__version__ = "0.1.0"


In [ ]:
%%writefile adaptmem/types.py
"""Shared dataclasses."""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import TYPE_CHECKING, Any

if TYPE_CHECKING:
    from sentence_transformers import InputExample


@dataclass
class LabelledQuery:
    """A user-supplied training signal: query + ids of corpus chunks that satisfy it."""

    query: str
    relevant_ids: list[str]
    """IDs of chunks/passages in your corpus that are relevant to this query."""


@dataclass
class RetrievalHit:
    chunk_id: str
    text: str
    score: float


@dataclass
class TrainingPair:
    """Output of hard-negative mining; consumed by the trainer."""

    anchor: str
    positive: str
    negative: str
    qid: str = ""
    pos_id: str = ""
    neg_id: str = ""

    def to_input_example(self) -> "InputExample":
        """Convert to sentence_transformers.InputExample (lazy import)."""
        from sentence_transformers import InputExample

        return InputExample(texts=[self.anchor, self.positive, self.negative])


@dataclass
class TrainConfig:
    """Knobs you'll tune. Defaults are the recipe that produced 0.9950 R@5 on LongMemEval."""

    epochs: int = 1
    batch_size: int = 8
    learning_rate: float = 2e-5
    warmup_ratio: float = 0.1
    top_k_mine: int = 10
    """How many top-K hits to inspect when picking a hard negative per (query, gt) pair."""
    extra_metadata: dict[str, Any] = field(default_factory=dict)


In [ ]:
%%writefile adaptmem/miner.py
"""Hard-negative mining over a corpus.

For each labelled query, we encode the corpus once with a base model, retrieve
the top-K candidates, and pick the first non-relevant one as a hard negative.
A "hard" negative shares lexical/semantic surface with the query but is
genuinely wrong — these are the examples a contrastive loss learns from.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from typing import Any

import numpy as np

from adaptmem.types import LabelledQuery, TrainingPair


@dataclass
class CorpusEntry:
    id: str
    text: str


class HardNegativeMiner:
    """Mines (anchor, positive, negative) triples from a corpus + labelled queries.

    Strategy:
    - Encode corpus once with a `base_model` (sentence-transformers SentenceTransformer).
    - For each query, retrieve top-K. Pick the first non-relevant id as the hard negative.
    - If all top-K are relevant, fall back to a uniformly-random non-relevant id.
    - One triple is emitted per (query, relevant_id) — multi-label queries spawn multiple
      training pairs, all anchored on the same query but with distinct positives.
    """

    def __init__(self, base_model: Any, top_k_mine: int = 10, seed: int = 42):
        self.base_model = base_model
        self.top_k_mine = top_k_mine
        self.rng = random.Random(seed)

    def mine(
        self, corpus: list[CorpusEntry], queries: list[LabelledQuery]
    ) -> list[TrainingPair]:
        if not corpus:
            return []
        # Encode corpus once
        ids = [c.id for c in corpus]
        texts = [c.text for c in corpus]
        embs = self.base_model.encode(
            texts,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        pairs: list[TrainingPair] = []
        for q in queries:
            qv = self.base_model.encode(
                [q.query],
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )[0]
            scores = embs @ qv  # (N,)
            k = min(self.top_k_mine, len(ids))
            top_idx = np.argpartition(-scores, k - 1)[:k]
            top_idx = top_idx[np.argsort(-scores[top_idx])]
            top_ids = [ids[i] for i in top_idx]

            relevant = set(q.relevant_ids)
            # Pick first non-relevant in top-K
            neg_idx = None
            for j, sid in zip(top_idx, top_ids):
                if sid not in relevant:
                    neg_idx = j
                    break

            if neg_idx is None:
                # Random non-relevant from full corpus
                non_rel_pool = [i for i, sid in enumerate(ids) if sid not in relevant]
                if not non_rel_pool:
                    continue
                neg_idx = self.rng.choice(non_rel_pool)

            neg_text = texts[neg_idx]
            neg_id = ids[neg_idx]

            id_to_text = {sid: txt for sid, txt in zip(ids, texts)}
            for rel_id in q.relevant_ids:
                if rel_id not in id_to_text:
                    continue
                pairs.append(
                    TrainingPair(
                        anchor=q.query,
                        positive=id_to_text[rel_id],
                        negative=neg_text,
                        pos_id=rel_id,
                        neg_id=neg_id,
                    )
                )

        return pairs


In [ ]:
%%writefile adaptmem/core.py
"""High-level AdaptMem class: train + persist + search."""
from __future__ import annotations

from pathlib import Path

import numpy as np

from adaptmem.miner import CorpusEntry, HardNegativeMiner
from adaptmem.types import LabelledQuery, RetrievalHit, TrainConfig


class AdaptMem:
    """One-shot domain adaptation for retrieval.

    Default flow:
        am = AdaptMem(base_model="all-MiniLM-L6-v2")
        am.train(corpus=[...], labelled=[LabelledQuery(...), ...])
        hits = am.search("question text", top_k=5)

    The base model is loaded lazily (first `train` or `load` call). The
    fine-tuned model lives in memory until you call `save(path)`.
    """

    def __init__(
        self,
        base_model: str = "all-MiniLM-L6-v2",
        rerank: bool = False,
        rerank_model: str = "cross-encoder/ms-marco-MiniLM-L-12-v2",
        device: str | None = None,
    ):
        self.base_model_name = base_model
        self._model = None
        self._corpus: list[CorpusEntry] = []
        self._embeddings: np.ndarray | None = None
        # Optional cross-encoder rerank stage. Disabled by default; when enabled,
        # `search` fetches a wider bi-encoder candidate set and reorders it with
        # the cross-encoder before returning top_k.
        self.rerank_enabled = rerank
        self.rerank_model_name = rerank_model
        self._rerank_model = None
        # Device override for the underlying SentenceTransformer. None lets
        # PyTorch auto-detect (CUDA → MPS → CPU). Passing "cpu" sidesteps
        # MPS deadlocks that have surfaced on some Apple-silicon configs
        # during contrastive fine-tuning. Persisted in config.json so that
        # .load() can restore the same choice.
        self.device = device

    # ---- Training -------------------------------------------------------
    def train(
        self,
        corpus: list[str] | list[CorpusEntry] | list[dict],
        labelled: list[LabelledQuery] | list[dict],
        config: TrainConfig | None = None,
    ) -> dict:
        """Mine hard negatives, fine-tune via MultipleNegativesRankingLoss, build index.

        `corpus` can be:
          - list[str] — auto-assigned ids "c0", "c1", ...
          - list[CorpusEntry]
          - list[dict] with keys {"id", "text"}

        Returns a dict with training stats (n_pairs, train_loss, runtime_s).
        """
        config = config or TrainConfig()
        entries = _normalise_corpus(corpus)
        queries = _normalise_queries(labelled)
        self._corpus = entries

        from sentence_transformers import SentenceTransformer

        st_kwargs = {"device": self.device} if self.device else {}
        base = SentenceTransformer(self.base_model_name, **st_kwargs)
        miner = HardNegativeMiner(base_model=base, top_k_mine=config.top_k_mine)
        pairs = miner.mine(entries, queries)
        if not pairs:
            raise ValueError("Hard-negative mining produced 0 pairs — check your labels.")

        # Fine-tune
        from sentence_transformers import losses
        from torch.utils.data import DataLoader

        examples = [p.to_input_example() for p in pairs]
        loader = DataLoader(examples, shuffle=True, batch_size=config.batch_size)
        loss = losses.MultipleNegativesRankingLoss(base)

        import time

        t0 = time.time()
        n_steps = max(1, (len(examples) // config.batch_size) * config.epochs)
        warmup = int(n_steps * config.warmup_ratio)
        base.fit(
            train_objectives=[(loader, loss)],
            epochs=config.epochs,
            warmup_steps=warmup,
            optimizer_params={"lr": config.learning_rate},
            show_progress_bar=False,
        )
        runtime = time.time() - t0

        self._model = base
        # Build index over the corpus with the freshly tuned model
        self._embeddings = base.encode(
            [c.text for c in entries],
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        # Approximate token count (whitespace heuristic — tokenizer-free so it
        # works for any base model). Word count is the common proxy in
        # retrieval logs; callers want an order-of-magnitude budget, not an
        # exact tokenizer count.
        all_texts: list[str] = [c.text for c in entries]
        for p in pairs:
            all_texts.append(p.anchor)
            all_texts.append(p.positive)
            all_texts.append(p.negative)
        n_tokens_approx = sum(len(t.split()) for t in all_texts)
        return {
            "n_pairs": len(pairs),
            "runtime_s": round(runtime, 2),
            "n_steps": n_steps,
            "n_tokens_approx": n_tokens_approx,
            "tokens_per_s": round(n_tokens_approx / runtime, 1) if runtime > 0 else 0.0,
        }

    # ---- Public introspection (stable contract for downstream tools) ---
    @property
    def encoder(self) -> Any:
        """The underlying SentenceTransformer (after .train() or .load()).

        Exposed so downstream packages (e.g. halluguard) can plug the tuned
        encoder into their own retrievers without reaching into `_model`.
        """
        return self._model

    @property
    def corpus(self) -> list[CorpusEntry]:
        """The indexed corpus entries, in insertion order."""
        return list(self._corpus)

    @property
    def embeddings(self) -> np.ndarray | None:
        """The L2-normalised embedding matrix aligned with `corpus`."""
        return self._embeddings

    # ---- Streaming corpus updates -------------------------------------
    def add_corpus(self, new_corpus: list[str] | list[CorpusEntry] | list[dict]) -> int:
        """Append entries to the in-memory index without re-encoding the
        whole corpus. Returns the number of newly added entries.

        Skips entries whose `id` already exists in the current index — safe
        to call repeatedly with overlapping batches. Existing embeddings
        are preserved; only the new chunk texts are encoded.

        Requires that `.train()` or `.load()` was called first (the index
        encoder + embedding matrix must already exist).
        """
        if self._model is None or self._embeddings is None:
            raise RuntimeError("Not initialised. Call .train() or .load() first.")
        new_entries = _normalise_corpus(new_corpus)
        existing_ids = {c.id for c in self._corpus}
        fresh = [e for e in new_entries if e.id not in existing_ids]
        if not fresh:
            return 0
        new_vecs = self._model.encode(
            [e.text for e in fresh],
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        self._corpus.extend(fresh)
        self._embeddings = np.vstack([self._embeddings, new_vecs])
        return len(fresh)

    # ---- Persistence ---------------------------------------------------
    def save(self, path: str | Path) -> None:
        if self._model is None:
            raise RuntimeError("No model to save. Call .train() or .load() first.")
        out = Path(path)
        out.mkdir(parents=True, exist_ok=True)
        self._model.save(str(out / "model"))
        # Persist corpus + embeddings for inference reload
        np.save(out / "embeddings.npy", self._embeddings)
        with open(out / "corpus.tsv", "w") as f:
            for c in self._corpus:
                # tab-safe: escape tabs/newlines in text
                t = c.text.replace("\t", " ").replace("\n", " ")
                f.write(f"{c.id}\t{t}\n")
        # Persist config so `.load(path)` restores rerank settings.
        import json as _json
        (out / "config.json").write_text(
            _json.dumps(
                {
                    "base_model": self.base_model_name,
                    "rerank": self.rerank_enabled,
                    "rerank_model": self.rerank_model_name,
                    "device": self.device,
                }
            )
        )

    @classmethod
    def load(cls, path: str | Path) -> "AdaptMem":
        from sentence_transformers import SentenceTransformer

        p = Path(path)
        am = cls.__new__(cls)
        am.base_model_name = ""
        am.device = None
        # Read device from config.json (if present) before constructing the
        # SentenceTransformer so we can honour an explicit "cpu" choice.
        cfg_path = p / "config.json"
        cfg = None
        if cfg_path.exists():
            import json as _json
            cfg = _json.loads(cfg_path.read_text())
            am.device = cfg.get("device")
        st_kwargs = {"device": am.device} if am.device else {}
        am._model = SentenceTransformer(str(p / "model"), **st_kwargs)
        am._embeddings = np.load(p / "embeddings.npy")
        am._corpus = []
        with open(p / "corpus.tsv") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                cid, text = line.split("\t", 1)
                am._corpus.append(CorpusEntry(id=cid, text=text))
        # Restore rerank config (cfg loaded above for device — reuse).
        if cfg is not None:
            am.base_model_name = cfg.get("base_model", "")
            am.rerank_enabled = bool(cfg.get("rerank", False))
            am.rerank_model_name = cfg.get(
                "rerank_model", "cross-encoder/ms-marco-MiniLM-L-12-v2"
            )
        else:
            am.rerank_enabled = False
            am.rerank_model_name = "cross-encoder/ms-marco-MiniLM-L-12-v2"
        am._rerank_model = None
        return am

    # ---- Inference -----------------------------------------------------
    def _ensure_rerank_model(self) -> None:
        if self._rerank_model is None:
            from sentence_transformers import CrossEncoder
            self._rerank_model = CrossEncoder(self.rerank_model_name)

    def search(
        self,
        query: str,
        top_k: int = 5,
        rerank_top_k: int | None = None,
    ) -> list[RetrievalHit]:
        """Retrieve top_k passages.

        When `rerank_enabled` is True, fetch `rerank_top_k or top_k * 3`
        candidates from the bi-encoder index, score the (query, candidate)
        pairs with the cross-encoder, and return the top_k by CE score.
        Cross-encoder is lazy-loaded.
        """
        if self._model is None or self._embeddings is None:
            raise RuntimeError("Not initialised. Call .train() or .load() first.")
        qv = self._model.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
        )[0]
        scores = self._embeddings @ qv
        candidate_k = (rerank_top_k or max(top_k * 3, top_k)) if self.rerank_enabled else top_k
        candidate_k = min(candidate_k, len(self._corpus))
        idx = np.argpartition(-scores, candidate_k - 1)[:candidate_k]
        idx = idx[np.argsort(-scores[idx])]

        if not self.rerank_enabled:
            return [
                RetrievalHit(
                    chunk_id=self._corpus[i].id,
                    text=self._corpus[i].text,
                    score=float(scores[i]),
                )
                for i in idx
            ]

        # Cross-encoder rerank
        self._ensure_rerank_model()
        candidates = [self._corpus[i] for i in idx]
        pairs = [(query, c.text) for c in candidates]
        ce_scores = self._rerank_model.predict(pairs, show_progress_bar=False)
        ranked = sorted(zip(candidates, ce_scores), key=lambda x: -float(x[1]))
        k = min(top_k, len(ranked))
        return [
            RetrievalHit(chunk_id=c.id, text=c.text, score=float(s))
            for c, s in ranked[:k]
        ]


# ---- helpers ---------------------------------------------------------
def _normalise_corpus(
    corpus: list[str] | list[CorpusEntry] | list[dict],
) -> list[CorpusEntry]:
    out: list[CorpusEntry] = []
    for i, c in enumerate(corpus):
        if isinstance(c, str):
            out.append(CorpusEntry(id=f"c{i}", text=c))
        elif isinstance(c, CorpusEntry):
            out.append(c)
        elif isinstance(c, dict):
            out.append(CorpusEntry(id=str(c.get("id", f"c{i}")), text=c["text"]))
        else:
            raise TypeError(f"corpus item {i} has unsupported type {type(c).__name__}")
    return out


def _normalise_queries(
    labelled: list[LabelledQuery] | list[dict],
) -> list[LabelledQuery]:
    out: list[LabelledQuery] = []
    for q in labelled:
        if isinstance(q, LabelledQuery):
            out.append(q)
        elif isinstance(q, dict):
            out.append(LabelledQuery(query=q["query"], relevant_ids=list(q["relevant_ids"])))
        else:
            raise TypeError(f"labelled item has unsupported type {type(q).__name__}")
    return out


In [ ]:
%%writefile benchmarks/longmemeval_eval.py
"""LongMemEval reproduction benchmark for adaptmem.

Reproduces the README's R@5=0.9950 claim on the held-out 200-question split.

Protocol (matches MemPalace + metis-pair eval_retrieval.py):
  - Per-question fresh corpus: each question's haystack_sessions is its own pool
  - User-only encoding: session = "\\n".join(t.content for t in turns if role=="user")
  - Empty sessions (no user turns) are dropped
  - relevant_ids = answer_session_ids
  - Recall@k: 1.0 if any GT id is in top-k, else 0.0; reported as macro-mean

Modes:
  --mode train   Mine + fine-tune on N train questions, save AdaptMem package.
                 Train hyperparams match the FT-300 recipe: epochs=3, batch=16, lr=2e-5.
  --mode test    Evaluate a saved adaptmem model on a held-out set.
                 If --st-model is given (a raw SentenceTransformer dir), evaluate
                 that directly using the same protocol — used to verify parity
                 with the standalone FT-300 model.

Splits:
  By default, uses the seed=42 shuffle from metis-pair/training_300:
    300 train / 200 test (matches the existing FT-300 model).
  Override with --split-ids PATH to a JSON {train_question_ids: [...], test_question_ids: [...]}.

CLI:
  longmemeval_eval.py --mode train --dataset longmemeval_s_cleaned.json \\
                      --split-ids split_ids.json --n-train 300 \\
                      --model-out ./bench-model

  longmemeval_eval.py --mode test  --dataset longmemeval_s_cleaned.json \\
                      --split-ids split_ids.json --model-in ./bench-model \\
                      [--results-out results.json]

  longmemeval_eval.py --mode test  --dataset longmemeval_s_cleaned.json \\
                      --split-ids split_ids.json --st-model ./minilm-lme-ft-300

No LLM. No API. CPU-friendly.
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path


def session_to_text(session_turns) -> str:
    """User-only encoding: concatenate user contents with newlines."""
    return "\n".join(
        t.get("content", "") for t in session_turns if t.get("role") == "user"
    )


def load_dataset(path: str) -> list[dict]:
    with open(path) as f:
        return json.load(f)


def build_question_index(data: list[dict]) -> dict[str, dict]:
    return {q["question_id"]: q for q in data}


def select_questions(
    data: list[dict], split_ids: dict | None, kind: str, n: int | None
) -> list[dict]:
    """kind: 'train' or 'test'. Returns the question dicts in their split-ID order."""
    if split_ids:
        key = "train_question_ids" if kind == "train" else "test_question_ids"
        ids = split_ids[key]
        if n is not None and kind == "train":
            ids = ids[:n]
        idx = build_question_index(data)
        return [idx[qid] for qid in ids if qid in idx]
    # Fallback: first N as train, rest as test
    if kind == "train":
        return data[: n or 300]
    return data[(n or 300):]


def make_per_question_corpus(q: dict) -> tuple[list[str], list[str]]:
    """Return (active_sids, active_docs) — non-empty sessions only, aligned."""
    sids = list(q["haystack_session_ids"])
    sessions = list(q["haystack_sessions"])
    docs = [session_to_text(s) for s in sessions]
    active = [(sid, d) for sid, d in zip(sids, docs) if d]
    if not active:
        return [], []
    active_sids, active_docs = zip(*active)
    return list(active_sids), list(active_docs)


# ---- Train mode ---------------------------------------------------------
def build_labelled_queries(train_questions: list[dict]) -> tuple[list[dict], list[dict]]:
    """Build (corpus_entries, labelled_queries) suitable for AdaptMem.train.

    The corpus is the union of all train haystack sessions across questions,
    keyed by session_id. Sessions with no user turns are dropped. Duplicate
    session_ids across questions are de-duped on first sight (LongMemEval
    haystacks are per-question disjoint in practice, but we guard anyway).
    """
    corpus_seen: dict[str, str] = {}
    labelled: list[dict] = []
    for q in train_questions:
        sids, docs = make_per_question_corpus(q)
        if not sids:
            continue
        for sid, doc in zip(sids, docs):
            if sid not in corpus_seen:
                corpus_seen[sid] = doc
        # Only keep relevant_ids that are actually present in the (filtered) corpus
        relevant = [sid for sid in q["answer_session_ids"] if sid in corpus_seen]
        if not relevant:
            continue
        labelled.append({"query": q["question"], "relevant_ids": relevant})
    corpus = [{"id": sid, "text": text} for sid, text in corpus_seen.items()]
    return corpus, labelled


def cmd_train(args) -> None:
    from adaptmem import AdaptMem
    from adaptmem.types import TrainConfig

    print(f"loading dataset {args.dataset}…", file=sys.stderr, flush=True)
    data = load_dataset(args.dataset)
    print(f"  {len(data)} questions total", file=sys.stderr, flush=True)

    split_ids = json.load(open(args.split_ids)) if args.split_ids else None
    train_qs = select_questions(data, split_ids, "train", args.n_train)
    print(f"  train questions: {len(train_qs)}", file=sys.stderr, flush=True)

    corpus, labelled = build_labelled_queries(train_qs)
    print(
        f"  corpus entries: {len(corpus)}, labelled queries: {len(labelled)}",
        file=sys.stderr,
        flush=True,
    )

    cfg = TrainConfig(
        epochs=args.epochs,
        batch_size=args.batch,
        learning_rate=args.lr,
        warmup_ratio=args.warmup_ratio,
        top_k_mine=args.top_k_mine,
    )
    am = AdaptMem(base_model=args.base_model, device=args.device)

    t0 = time.time()
    stats = am.train(corpus=corpus, labelled=labelled, config=cfg)
    total = time.time() - t0

    out = Path(args.model_out)
    am.save(out)
    stats["wall_clock_s"] = round(total, 2)
    stats["n_train_questions"] = len(train_qs)
    stats["n_corpus"] = len(corpus)
    stats["epochs"] = cfg.epochs
    stats["batch_size"] = cfg.batch_size
    stats["learning_rate"] = cfg.learning_rate
    stats["base_model"] = args.base_model
    print(json.dumps(stats, indent=2))
    (out / "train_stats.json").write_text(json.dumps(stats, indent=2))


# ---- Test mode ---------------------------------------------------------
def recall_at_k(retrieved_ids: list[str], gt: set[str], k: int) -> float:
    return 1.0 if any(g in retrieved_ids[:k] for g in gt) else 0.0


def evaluate_with_st_model(model, test_qs: list[dict], top_k: int = 10) -> dict:
    """Per-question evaluation using a raw SentenceTransformer model.

    Returns macro Recall@1/5/10 plus a per-question-type breakdown
    (matches MemPal's published table layout).
    """
    import numpy as np

    n = len(test_qs)
    r1 = r5 = r10 = 0.0
    skipped = 0
    # Per-question-type counters: type -> {n, r1, r5, r10}
    per_type: dict[str, dict[str, float]] = {}
    t0 = time.time()
    for i, q in enumerate(test_qs):
        sids, docs = make_per_question_corpus(q)
        if not sids:
            skipped += 1
            continue
        embs = model.encode(
            docs,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        qv = model.encode(
            [q["question"]],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]
        scores = embs @ qv
        order = np.argsort(-scores)
        ranked_sids = [sids[j] for j in order]
        gt = set(q["answer_session_ids"])
        h1 = recall_at_k(ranked_sids, gt, 1)
        h5 = recall_at_k(ranked_sids, gt, 5)
        h10 = recall_at_k(ranked_sids, gt, 10)
        r1 += h1
        r5 += h5
        r10 += h10
        qtype = q.get("question_type", "unknown")
        bucket = per_type.setdefault(qtype, {"n": 0, "r1": 0.0, "r5": 0.0, "r10": 0.0})
        bucket["n"] += 1
        bucket["r1"] += h1
        bucket["r5"] += h5
        bucket["r10"] += h10
        if (i + 1) % 50 == 0:
            elapsed = time.time() - t0
            eta = elapsed * (n - i - 1) / (i + 1)
            print(f"  q{i+1}/{n}  eta {eta:.0f}s", file=sys.stderr, flush=True)
    n_eff = n - skipped
    per_type_summary = {
        t: {
            "n": int(b["n"]),
            "r1": round(b["r1"] / b["n"], 4) if b["n"] else 0.0,
            "r5": round(b["r5"] / b["n"], 4) if b["n"] else 0.0,
            "r10": round(b["r10"] / b["n"], 4) if b["n"] else 0.0,
        }
        for t, b in sorted(per_type.items())
    }
    return {
        "n_test": n,
        "n_evaluated": n_eff,
        "n_skipped_empty": skipped,
        "r1": round(r1 / n, 4) if n else 0.0,
        "r5": round(r5 / n, 4) if n else 0.0,
        "r10": round(r10 / n, 4) if n else 0.0,
        "per_question_type": per_type_summary,
        "wall_clock_s": round(time.time() - t0, 2),
    }


def cmd_test(args) -> None:
    print(f"loading dataset {args.dataset}…", file=sys.stderr, flush=True)
    data = load_dataset(args.dataset)
    split_ids = json.load(open(args.split_ids)) if args.split_ids else None
    test_qs = select_questions(data, split_ids, "test", args.n_train)
    print(f"  test questions: {len(test_qs)}", file=sys.stderr, flush=True)

    if args.st_model:
        from sentence_transformers import SentenceTransformer

        print(
            f"  loading raw SentenceTransformer {args.st_model}…",
            file=sys.stderr,
            flush=True,
        )
        model = SentenceTransformer(args.st_model)
        results = evaluate_with_st_model(model, test_qs, top_k=10)
        results["mode"] = "st-model"
        results["model_path"] = args.st_model
    else:
        if not args.model_in:
            print("--model-in required when --st-model is not used", file=sys.stderr)
            sys.exit(2)
        from adaptmem import AdaptMem
        from sentence_transformers import SentenceTransformer

        print(f"  loading adaptmem model {args.model_in}…", file=sys.stderr, flush=True)
        am = AdaptMem.load(args.model_in)
        # We bypass AdaptMem.search per-question because its index is the global
        # train corpus, while LongMemEval evaluation needs a per-question fresh
        # corpus. We reuse only the underlying SentenceTransformer.
        model: SentenceTransformer = am._model  # noqa: SLF001
        results = evaluate_with_st_model(model, test_qs, top_k=10)
        results["mode"] = "adaptmem"
        results["model_path"] = args.model_in

    results["dataset"] = args.dataset
    results["split_ids"] = args.split_ids
    print()
    print(f"# Mode: {results['mode']}")
    print(f"# Model: {results['model_path']}")
    print(f"# n_test={results['n_test']} (skipped_empty={results['n_skipped_empty']})")
    print(f"# R@1={results['r1']}  R@5={results['r5']}  R@10={results['r10']}")

    if args.results_out:
        Path(args.results_out).write_text(json.dumps(results, indent=2))
        print(f"# wrote {args.results_out}", file=sys.stderr)


# ---- Main ---------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--mode", required=True, choices=["train", "test"])
    ap.add_argument(
        "--dataset",
        default="/Users/macmini/Projects/metis-pair/benchmarks/data/longmemeval/longmemeval_s_cleaned.json",
        help="Path to longmemeval_s_cleaned.json",
    )
    ap.add_argument(
        "--split-ids",
        default="/Users/macmini/Projects/metis-pair/benchmarks/data/training_300/split_ids.json",
        help="Path to split_ids.json with train/test question_id lists",
    )
    ap.add_argument("--n-train", type=int, default=300, help="Number of train questions")
    # train-only
    ap.add_argument("--model-out", default="./bench-model", help="Output dir (train)")
    ap.add_argument("--base-model", default="all-MiniLM-L6-v2")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--warmup-ratio", type=float, default=0.1)
    ap.add_argument("--top-k-mine", type=int, default=10)
    # test-only
    ap.add_argument("--model-in", help="Adaptmem model dir (test)")
    ap.add_argument(
        "--st-model",
        help="Raw SentenceTransformer dir (test) — used to evaluate the existing FT-300 model directly",
    )
    ap.add_argument("--results-out", help="Write results JSON here")
    ap.add_argument(
        "--device",
        default=None,
        help="Force PyTorch device: 'cpu', 'cuda', 'mps'. Default = autodetect. "
             "Pass 'cpu' to bypass MPS deadlocks observed on Apple silicon.",
    )
    args = ap.parse_args()

    if args.mode == "train":
        cmd_train(args)
    else:
        cmd_test(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile benchmarks/data/split_ids_100_400.json
{
  "train_question_ids": [
    "cc06de0d",
    "f9e8c073",
    "b320f3f8",
    "a89d7624",
    "311778f1",
    "gpt4_59c863d7",
    "bbf86515",
    "099778bb",
    "e831120c",
    "dcfa8644",
    "8fb83627",
    "e66b632c",
    "gpt4_7fce9456",
    "55241a1f",
    "352ab8bd",
    "f4f1d8a4",
    "830ce83f",
    "2311e44b",
    "09ba9854",
    "gpt4_a1b77f9c",
    "07741c45",
    "gpt4_70e84552",
    "b46e15ee",
    "6071bd76",
    "6f9b354f",
    "1d4da289",
    "gpt4_8279ba02",
    "6456829e_abs",
    "0db4c65d",
    "d6062bb9",
    "60bf93ed_abs",
    "d3ab962e",
    "87f22b4a",
    "e01b8e2f",
    "gpt4_7ddcf75f",
    "8ebdbe50",
    "26bdc477",
    "29f2956b_abs",
    "2311e44b_abs",
    "75f70248",
    "852ce960",
    "f0e564bc",
    "fca70973",
    "3c1045c8",
    "18bc8abd",
    "afdc33df",
    "54026fce",
    "b9cfe692",
    "6456829e",
    "e6041065",
    "gpt4_15e38248",
    "gpt4_2ba83207",
    "2133c1b5_abs",
    "gpt4_8279ba03",
    "76d63226",
    "1192316e",
    "gpt4_fa19884d",
    "gpt4_372c3eed_abs",
    "1a8a66a6",
    "gpt4_fe651585",
    "e25c3b8d",
    "945e3d21",
    "86b68151",
    "1c0ddc50",
    "1e043500",
    "d682f1a2",
    "gpt4_b5700ca0",
    "91b15a6e",
    "ce6d2d27",
    "f523d9fe",
    "7024f17c",
    "8752c811",
    "gpt4_f420262d",
    "d01c6aa8",
    "4b24c848",
    "7e974930",
    "3fdac837",
    "gpt4_b4a80587",
    "c18a7dc8",
    "80ec1f4f_abs",
    "7527f7e2",
    "6ade9755",
    "89941a94",
    "gpt4_1d80365e",
    "2133c1b5",
    "06db6396",
    "gpt4_88806d6e",
    "88432d0a",
    "3ba21379",
    "0862e8bf",
    "aae3761f",
    "5025383b",
    "gpt4_e061b84f",
    "73d42213",
    "4bc144e2",
    "gpt4_5501fe77",
    "00ca467f",
    "dfde3500",
    "01493427",
    "b6025781"
  ],
  "test_question_ids": [
    "a96c20ee_abs",
    "982b5123_abs",
    "gpt4_fa19884c",
    "gpt4_1a1dc16d",
    "28dc39ac",
    "gpt4_2d58bcd6",
    "51c32626",
    "c4ea545c",
    "1da05512",
    "gpt4_385a5000",
    "577d4d32",
    "72e3ee87",
    "f4f1d8a4_abs",
    "9d25d4e0",
    "b29f3365",
    "b759caee",
    "10e09553",
    "1d4e3b97",
    "d52b4f67",
    "gpt4_e072b769",
    "58ef2f1c",
    "6e984301",
    "41275add",
    "gpt4_59149c77",
    "2ebe6c90",
    "1cea1afa",
    "gpt4_1e4a8aec",
    "6c49646a",
    "8a2466db",
    "gpt4_65aabe59",
    "gpt4_93159ced",
    "51a45a95",
    "af8d2e46",
    "561fabcd",
    "370a8ff4",
    "gpt4_d84a3211",
    "gpt4_7a0daae1",
    "2a1811e2",
    "gpt4_78cf46a3",
    "1568498a",
    "6b7dfb22",
    "6ae235be",
    "bc8a6e93_abs",
    "681a1674",
    "06878be2",
    "1a1907b4",
    "0e4e4c46",
    "gpt4_85da3956",
    "gpt4_f420262c",
    "2bf43736",
    "bc149d6b",
    "09d032c9",
    "5c40ec5b",
    "eac54adc",
    "993da5e2",
    "71a3fd6b",
    "gpt4_0b2f1d21",
    "ad7109d1",
    "4c36ccef",
    "c8c3f81d",
    "edced276_abs",
    "0bc8ad92",
    "gpt4_468eb064",
    "2ebe6c92",
    "cc6d1ec1",
    "4dfccbf8",
    "95228167",
    "ba358f49",
    "45dc21b6",
    "db467c8c",
    "720133ac",
    "67e0d0f2",
    "cc5ded98",
    "726462e0",
    "4100d0a0",
    "3a704032",
    "gpt4_7ca326fa",
    "ec81a493",
    "618f13b2",
    "58470ed2",
    "gpt4_4fc4f797",
    "60036106",
    "157a136e",
    "6222b6eb",
    "69fee5aa",
    "19b5f2b3_abs",
    "gpt4_d12ceb0e",
    "51b23612",
    "2318644b",
    "3fe836c9",
    "gpt4_7de946e7",
    "71017277",
    "f0853d11",
    "dc439ea3",
    "gpt4_2f91af09",
    "9a707b81",
    "bc8a6e93",
    "c14c00dd",
    "8979f9ec",
    "cf22b7bf",
    "gpt4_ec93e27f",
    "gpt4_468eb063",
    "41698283",
    "1de5cff2",
    "21d02d0d",
    "c7cf7dfd",
    "gpt4_ab202e7f",
    "dccbc061",
    "078150f1",
    "e3038f8c",
    "gpt4_c27434e8_abs",
    "2698e78f",
    "031748ae_abs",
    "gpt4_59149c78",
    "c8f1aeed",
    "184da446",
    "gpt4_b5700ca9",
    "89527b6b",
    "0977f2af",
    "853b0a1d",
    "a346bb18",
    "3249768e",
    "gpt4_2f8be40d",
    "gpt4_93159ced_abs",
    "eeda8a6d",
    "7a8d0b71",
    "95bcc1c8",
    "gpt4_2487a7cb",
    "85fa3a3f",
    "7e00a6cb",
    "e3fc4d6e",
    "59524333",
    "37f165cf",
    "0ddfec37",
    "60bf93ed",
    "d7c942c3",
    "80ec1f4f",
    "ceb54acb",
    "9aaed6a3",
    "gpt4_4929293a",
    "ed4ddc30",
    "545bd2b5",
    "2788b940",
    "ef9cf60a",
    "gpt4_7f6b06db",
    "0ea62687",
    "3d86fd0a",
    "3e321797",
    "d24813b1",
    "38146c39",
    "efc3f7c2",
    "7401057b",
    "5809eb10",
    "28bcfaac",
    "1903aded",
    "gpt4_194be4b3",
    "gpt4_e414231f",
    "0ddfec37_abs",
    "c2ac3c61",
    "gpt4_4ef30696",
    "1f2b8d4f",
    "0f05491a",
    "8550ddae",
    "8077ef71",
    "b86304ba",
    "e61a7584",
    "8cf51dda",
    "gpt4_2f584639",
    "08e075c7",
    "5d3d2817",
    "7405e8b1",
    "a3045048",
    "gpt4_731e37d7",
    "c8090214_abs",
    "36580ce8",
    "ba358f49_abs",
    "gpt4_d6585ce8",
    "e56a43b9",
    "2c63a862",
    "gpt4_5438fa52",
    "07b6f563",
    "gpt4_31ff4165",
    "0bb5a684",
    "71315a70",
    "gpt4_cd90e484",
    "gpt4_8c8961ae",
    "gpt4_fe651585_abs",
    "36b9f61e",
    "gpt4_b0863698",
    "gpt4_1d4ab0c9",
    "15745da0_abs",
    "0862e8bf_abs",
    "bcbe585f",
    "a2f3aa27",
    "gpt4_6dc9b45b",
    "ccb36322",
    "f685340e",
    "9ea5eabc",
    "gpt4_372c3eed",
    "37d43f65",
    "bf659f65",
    "b0479f84",
    "gpt4_213fd887",
    "e4e14d04",
    "f8c5f88b",
    "gpt4_18c2b244",
    "a11281a2",
    "gpt4_2655b836",
    "e47becba",
    "gpt4_74aed68e",
    "gpt4_af6db32f",
    "6cb6f249",
    "77eafa52",
    "gpt4_93f6379c",
    "e8a79c70",
    "7a87bd0c",
    "gpt4_6ed717ea",
    "d6233ab6",
    "c19f7a0b",
    "gpt4_61e13b3c",
    "d23cf73b",
    "gpt4_1e4a8aeb",
    "ba61f0b9",
    "118b2229",
    "488d3006",
    "c4a1ceb8",
    "8e91e7d9",
    "42ec0761",
    "65240037",
    "fea54f57",
    "c8090214",
    "b01defab",
    "6aeb4375_abs",
    "faba32e5",
    "c5e8278d",
    "gpt4_e414231e",
    "eeda8a6d_abs",
    "gpt4_8e165409",
    "af082822",
    "22d2cb42",
    "92a0aa75",
    "1c549ce4",
    "25e5aa4f",
    "gpt4_68e94288",
    "4baee567",
    "18dcd5a5",
    "dad224aa",
    "gpt4_f2262a51",
    "29f2956b",
    "21436231",
    "19b5f2b3",
    "gpt4_1916e0ea",
    "gpt4_45189cb4",
    "0a995998",
    "b6019101",
    "9bbe84a2",
    "61f8c8f8",
    "9a707b82",
    "8cf4d046",
    "eac54add",
    "75832dbd",
    "gpt4_98f46fc6",
    "d596882b",
    "88432d0a_abs",
    "16c90bf4",
    "f685340e_abs",
    "b5ef892d",
    "gpt4_f49edff3",
    "gpt4_483dd43c",
    "bb7c3b45",
    "gpt4_7abb270c",
    "gpt4_9a159967",
    "07741c44",
    "4d6b87c8",
    "6aeb4375",
    "gpt4_d6585ce9",
    "60472f9c",
    "caf9ead2",
    "32260d93",
    "60159905",
    "0a34ad58",
    "a40e080f",
    "10d9b85a",
    "a06e4cfe",
    "4f54b7c9",
    "6613b389",
    "70b3e69b",
    "gpt4_7bc6cf22",
    "gpt4_0a05b494",
    "778164c6",
    "195a1a1b",
    "8464fc84",
    "b46e15ed",
    "603deb26",
    "eaca4986",
    "2698e78f_abs",
    "gpt4_21adecb5",
    "2e6d26dc",
    "5831f84d",
    "08f4fc43",
    "3f1e9474",
    "c9f37c46",
    "gpt4_2f56ae70",
    "1b9b7252",
    "35a27287",
    "gpt4_d31cdae3",
    "129d1232",
    "4adc0475",
    "27016adc",
    "46a3abf7",
    "9ee3ecd6",
    "982b5123",
    "09ba9854_abs",
    "0e5e2d1a",
    "e9327a54",
    "86f00804",
    "e982271f",
    "7161e7e2",
    "57f827a0",
    "6a27ffc2",
    "edced276",
    "gpt4_d9af6064",
    "75499fd8",
    "60d45044",
    "gpt4_70e84552_abs",
    "2ce6a0f2",
    "gpt4_4929293b",
    "a1cc6108",
    "gpt4_5dcc0aab",
    "a3838d2b",
    "c7dc5443",
    "505af2f5",
    "gpt4_68e94287",
    "15745da0",
    "0100672e",
    "a82c026e",
    "5e1b23de",
    "71017276",
    "89941a93",
    "6b168ec8",
    "affe2881",
    "0edc2aef",
    "gpt4_2312f94c",
    "a4996e51",
    "c6853660",
    "ef66a6e5",
    "8a137a7f",
    "a96c20ee",
    "fca762bc",
    "ac031881",
    "d905b33f",
    "e493bb7c",
    "a9f6b44c",
    "dd2973ad",
    "8aef76bc",
    "f35224e0",
    "8b9d4367",
    "gpt4_c27434e8",
    "gpt4_a56e767c",
    "eace081b",
    "5a4f22c0",
    "58bf7951",
    "c4f10528",
    "50635ada",
    "06f04340",
    "0bc8ad93",
    "e5ba910e_abs",
    "5a7937c8",
    "a3332713",
    "4388e9dd",
    "8c18457d",
    "gpt4_2c50253f",
    "6a1eabeb",
    "b3c15d39",
    "gpt4_e061b84g",
    "3b6f954b",
    "gpt4_76048e76",
    "4dfccbf7",
    "2b8f3739",
    "d851d5ba",
    "4fd1909e",
    "94f70d80",
    "66f24dbb",
    "a08a253f",
    "6e984302",
    "001be529",
    "gpt4_a2d1d1f6",
    "cc539528",
    "e48988bc",
    "gpt4_4cd9eba1",
    "8e9d538c",
    "a1eacc2a",
    "6d550036",
    "gpt4_e05b82a6",
    "81507db6",
    "caf03d32",
    "031748ae",
    "c960da58",
    "1faac195",
    "gpt4_4edbafa2"
  ]
}

In [ ]:
import os, shutil
src = '/content/longmemeval_s_cleaned.json'
dst = '/content/benchmarks/data/longmemeval_s_cleaned.json'
assert os.path.exists(src), 'longmemeval_s_cleaned.json yok — Files panelinden upload et'
if not os.path.exists(dst):
    shutil.copy(src, dst)
print('dataset:', os.path.getsize(dst) // 1024 // 1024, 'MB')

In [ ]:
import sys
if '/content' not in sys.path: sys.path.insert(0, '/content')
import adaptmem
print('adaptmem version:', adaptmem.__version__)

In [ ]:
# TRAIN
import os, subprocess
env = {**os.environ, 'PYTHONPATH': '/content'}
r = subprocess.run(['python', 'benchmarks/longmemeval_eval.py', '--mode', 'train', '--dataset', 'benchmarks/data/longmemeval_s_cleaned.json', '--split-ids', 'benchmarks/data/split_ids_100_400.json', '--n-train', '100', '--epochs', '1', '--device', 'cpu', '--model-out', 'benchmarks/bench-model-100'], env=env)
print('exit:', r.returncode)

In [ ]:
# TEST
import subprocess
r = subprocess.run(['python', 'benchmarks/longmemeval_eval.py', '--mode', 'test', '--dataset', 'benchmarks/data/longmemeval_s_cleaned.json', '--split-ids', 'benchmarks/data/split_ids_100_400.json', '--st-model', 'benchmarks/bench-model-100/model', '--device', 'cpu', '--results-out', 'benchmarks/results_ft100_400.json'])
print('exit:', r.returncode)

In [ ]:
import json
with open('benchmarks/results_ft100_400.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
from google.colab import files
files.download('benchmarks/results_ft100_400.json')